# Example 6.4 / 2: Solving the Continuous Relaxation

Companion notebook for **6.4 Fixing and Relaxing Variables** in the AMPL lab Documentation, section "Solving the Continuous Relaxation". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Solving the Continuous Relaxation

The continuous relaxation enlarges an integer model's feasible set. It therefore gives an upper
bound for a maximization problem and a lower bound for a minimization problem, provided the
relaxed problem is solved to optimality.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;
param profit {PROD};
param hours > 0;
param capacity >= 0;
var Make {PROD} integer >= 0, <= 4;
maximize TotalProfit: sum {p in PROD} profit[p] * Make[p];
subject to Limit: sum {p in PROD} hours * Make[p] <= capacity;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["profit"] = {"bands": 25, "coils": 30, "plate": 29}
ampl.param["hours"] = 2.5
ampl.param["capacity"] = 19

In [5]:
# run.py
ampl.option["solver_msg"] = 0
ampl.option["relax_integrality"] = 1
ampl.solve(solver="highs")
print("relaxation:", ampl.obj["TotalProfit"].value())
ampl.option["relax_integrality"] = 0
ampl.solve(solver="highs")
print("integer model:", ampl.obj["TotalProfit"].value())

relaxation: 224.4
integer model: 207.0


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := bands coils plate;
param profit := bands 25  coils 30  plate 29;
param hours := 2.5;
param capacity := 19;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
option solver_msg 0;
option relax_integrality 1;
solve;
printf "relaxation: %g\n", TotalProfit;
option relax_integrality 0;
solve;
printf "integer model: %g\n", TotalProfit;

relaxation: 224.4
integer model: 207
